<h1>Each Degrees website extractor</h1>

In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
def extract_content_and_tables(url, website_name=""):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.RequestException as e:
        return [{"metadata": website_name, "title": "Error", "content": str(e)}]

    soup = BeautifulSoup(response.text, "html.parser")
    content_list = []

    # Extract content under headings
    for heading in soup.find_all(['h2', 'h3', 'h4']):
        content = []
        for sibling in heading.find_next_siblings():
            if sibling.name and sibling.name.startswith('h'):
                break
            content.append(sibling.get_text(" ", strip=True))  # Extract text with spaces for better formatting
        
        if content:
            formatted_content = " ".join(content).replace("\n", " ").strip()
            content_list.append({
                "metadata": website_name,
                "title": heading.get_text(strip=True),
                "content": formatted_content
            })

    # Extract structured data from unstructured sections (e.g., Degree Structure)
    for div in soup.find_all("div", class_="some-class-for-degree-structure"):  # Replace with actual class name if known
        raw_text = div.get_text(" ", strip=True)  # Extract entire block text
        structured_text = (
            raw_text.replace("Department", "Department: ")
                    .replace("Level", "Level: ")
                    .replace("Study System", "Study System: ")
                    .replace("Total Credit Hours", "Total Credit Hours: ")
                    .replace("Duration", "Duration: ")
                    .replace("Intake", "Intake: ")
                    .replace("Language", "Language: ")
                    .replace("Study Mode", "Study Mode: ")
        )  # Add colons for clarity

        content_list.append({
            "metadata": website_name,
            "title": "Degree Structure",
            "content": structured_text
        })

    # Extract tables along with their titles (ul > li > strong)
    for li in soup.find_all("li"):
        strong_tag = li.find("strong")
        if strong_tag:
            table_title = strong_tag.get_text(strip=True)
            next_elements = strong_tag.find_all_next(["table", "p"])
            
            table_data = []
            for elem in next_elements:
                if elem.name == "table":
                    rows = []
                    for row in elem.find_all("tr"):
                        cells = [cell.get_text(strip=True) for cell in row.find_all(["th", "td"])]
                        rows.append(", ".join(cells))
                    if rows:
                        table_data.append(" || ".join(rows))

            # Each table gets its own dictionary with the same metadata and title
            for table in table_data:
                content_list.append({
                    "metadata": website_name,
                    "title": table_title,
                    "content": table
                })
    
    # Extract links for Study Plan PDFs or similar sections
    for a_tag in soup.find_all("a", href=True):
        href = a_tag["href"]
        if "study-plan" in href.lower() or "pdf" in href.lower():  # You can adjust the keyword based on the actual URL pattern
            if href.startswith("http"):  # Absolute URL
                pdf_url = href
            else:  # Relative URL, combine with the base URL
                pdf_url = requests.compat.urljoin(url, href)
            
            content_list.append({
                "metadata": website_name,
                "title": "Study Plan PDF",
                "content": pdf_url
            })

    return content_list


In [ ]:
url = "https://www.sharjah.ac.ae/en/Academics/Degree/Undergraduate/Information-Technology-Multimedia"
website_name = "Department of Computer Science :Bachelor of Science in Information Technology Multimedia"
extracted_data = extract_content_and_tables(url, website_name)

for item in extracted_data:
    print("Metadata:",item['metadata'], item['title'], "|| content:", item['content'])

Metadata: Department of Computer Science :Master of Science in Data Science Degree Structure || content: College Computing and Informatics Department Computer Science Level Graduate Masters Study System Courses and Theses Total Credit Hours 33 Cr. Hrs. Duration 2-4 Years Intake Fall and Spring Language English Study Mode Full Time and Part Time Begin your academic journey with our user-friendly online application platform. Apply Online
Metadata: Department of Computer Science :Master of Science in Data Science Important Dates || content: Graduate Studies Admission Deadline Graduate Studies Admission Deadline
Metadata: Department of Computer Science :Master of Science in Data Science Program Structure & Requirements || content: The MSc in Data Science program will stimulate research activities in the area of computer science, statistics, and their applications. This will enrich both undergraduate and graduate programs in both Colleges (College of Computing and Informatics and College of

<h1>Prof. and academic stuff Info getter</h1>

In [55]:
def extract_prof_info(url_data): 
    resp = requests.get(url_data, timeout=60)
    soup = BeautifulSoup(resp.text, 'html.parser')
    lineData = ""
    
    
    
    for k1 in soup.find_all("h1", class_="HomeBanner_h1__0zLcq"):
        name = k1.get_text(strip=True)
        lineData += name
        
    for k1 in soup.find_all("h4", class_="facultyAndStaffDirectoryDetailComp_listTitle__94bUV"):
        title = k1.get_text(strip=True)
        lineData = lineData+"("+ title+"). "
        
    for k1 in soup.find_all("li", class_="facultyAndStaffDirectoryDetailComp_listItem__jg8_H"):
        d1_elem = k1.find('h4')
        d2_elem = k1.find("p")
        if d1_elem is not None:
            d1 = d1_elem.get_text(strip=True)
            d2 = d2_elem.get_text(strip=True)
            lineData =lineData+ d1 + ":"+d2 + ". "
        
        # finding faculty details
        d = k1.find("a", href=True)
        det = "Faculty details: "
        if d is not None: 
            det = det+ d["href"] + ","
        if det != "Faculty details: ":
            lineData += det + ". "
    
    
    return lineData
    

In [ ]:
prof_urls = ["https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abbes-Amira", 
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hussein-M-Elmehdi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adnan-Sarhan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Esam-Agamy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hussein-M-Elmehdi", 
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ilhan-Ozturk",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jairo-Lugo-Ocando",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Karim-El-Zu-bi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kotb-Rissouni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamad-Alameddine",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadia-M-Alhasani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nouar-Tabet",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Qutayba-Hamid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Raafat-Abd-El-Gawad-El-Gharib-El-Awady",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sausan-Al-Kawas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zahia-Smail-Salhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amina-Mohammed-Almarzouqi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maamar-Bettayeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yousef-Haik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yousef-S-Haik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Farah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Bou-Nassif",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eman-Abu-Gharbieh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ihsan-Ahmed-Shehadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Saad-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kathafi-Izzat-AlGHananeem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maamoun-Saleh-Abdulkarim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Adel-Serhani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Maalej",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Al-Hawari",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tareq-Osaili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdeleziz-Tlili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bassel-Soudan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huseyin-Ozan-Tekin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khalil-Abdelrazek-Khalil-Abdelmawgoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Ben-Moussa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/MOURAD-BENSEGHIR",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wael-Rashdan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zahi-Badran",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/A-Aziz-Jaafar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdel-Nasser-Metwally-Aly-Kawde",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Hai-Al-Alami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Farouk-Radwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Cheaitou",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amr-Mohamed-Elnadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Andrew-Joseph-Power",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Arkan-Sam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Atidel-Aboubaker-Ben-Hadj-Alouane",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ayman-Fathy-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bashar-Afif-Issa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bushra-Ahmed-Alakashee",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Emad-S-Mushtaha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Firas-Ghanim-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghazi-Gaseem-Khaleel-Al-Khateeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamid-Alhaj",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamzah-Alzubaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hasan-Salem-Habshan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Heba-Hesham-Ali-Hijazi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huseyin-Seker",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Moustafa-Moustafa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Izmer-Bin-Ahmad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kareem-Mosa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Lucy-Semerjian",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manal-A-Awad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-saeed-Balajeed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Saad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Shikh-Abubaker-Albaity",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-alemoush",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Abdel-Karim-M-Al-Hourani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Ibrahim-El-Gamal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mounir-Kaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mutasem-Mark-RAWAS-QALAJI",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nabeel-Al-Yateem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadia-Khalifa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadjib-Benkheira",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reyad-Shaker-Obaid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saleh-Al-Luhaibi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sebti-Foufou",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaker-Jamal-Saleh-Bani-Melhem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shyamal-Kataria",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syarif-Junaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Thouraya-Snoussi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wael-Ahmed-Allam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wael-Ahmed-Allam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wael-M-Abdel-Rahman-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wael-Mohamed-Talaat-Taha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Waseem-El-Huneidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/AB-Rani-Samsudin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdal-Samee-Al-Aniess",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdallah-Shanbleh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdel-Nasser-Ahmed-El-Shorbagi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdelaziz-Soufyane",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abduelmula-Rajab-Abduelkarem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Ghani-Olabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Kadir-Hamid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abir-Jaafar-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adam-Bin-Husein",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-A-Al-Kubise",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Bouridane",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Elwakil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Falah-Alomosh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-M-Khedr",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Osama-Murad-Jamleh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ala-a-Yakoob-yousif",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alex-Opoku",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-A-El-Moursy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Ahmad-Al-Barakat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Al-Keblawy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-JabAllah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Ouni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Tourki",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amine-Ahriche",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ammar-Alkhalidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amr-A-Amin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Andreas-Rechkemmer",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anis-Allagui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anwar-Hasan-Jarndal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ashraf-Elnagar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ayssar-Nahl",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Basem-S-Attili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Basema-Saddik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Basharia-Yousef",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bashir-Mohamed-Suleiman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Belkacem-Said-Houari",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Benissa-Bettaher",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Chaouki-Ghenai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Elsiddig-Ahmed-Elmustafa-Elsheikh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ensanya-Ali-El-Saed-Abou-Neel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Farah-Naja",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Faridahwati-Mohd-Shamsudin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatih-Kurugollu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatma-A-Hegazy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fikri-Dweiri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Gehad-Sadiek",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghaleb-A-Rabab-ah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Gopinath-Vellore-Kannan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hafid-Ismaili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamdi-Bashir",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hany-A-Omar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hassan-Abdelhamid-Mahmoud-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hassan-M-El-Rifai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hichem-Eleuch",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hissam-Tawfik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hussain-M-Al-Othman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-I-El-Sharkawy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Kamel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ideisan-Abu-Abdoun",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ihab-Mohammad-Naji-Obaidat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ihsan-Ali-Khlaif-Al-Mahasenh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ilias-Fernini",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Imad-Alsyouf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Imad-Eldin-Abdul-Hay",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Talaat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ioannis-Savvaidis",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ismail-Saadoun",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ismail-Shahin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iyad-Jadalhaq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jason-Gainous",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/John-Rice",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kais-Daoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kassem-Saad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khaled-Hamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Leon-Barkho",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/M-Azhar-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/M-Munis-Ahmed-Awad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maha-M-Saber-Ayad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maher-Omar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-bin-Khalifa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Fayyad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mamdouh-El-Haj-Assad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manal-Marwan-Munajjed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mashhoor-Ahmad-Salameh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Matloub-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mawieh-Hamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mehmet-Omer-Gorduysus",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mehmood-Khan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mesut-Idriz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Gamal-Aboelmaged",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Hassan-Mohamed-Abdelazim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-S-AL-Hajjaj",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Saleh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Semai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Suliman-Elnor",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Ali-Abdelkareem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-AlShboul",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Harb-Semreen",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Shamsuzzaman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohd-Sobri-Takriff",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Monther-Abdeljabbar-Khanfar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mostafa-Zahri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Moussa-Leblouba",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muataz-Ali-Atieh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Islam-Mustafa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Naser-Khaled-Hasan-Nawayseh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Natheer-Hashim-Al-Rawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nezar-Atalla-Hammouri",
            "https://www.sharjah.ac.ae/en/Research/Our-Team/Nezar-Atalla-Hammouri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noha-Mellor",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Oguz-Ergin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ola-B-Al-Batayneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Qassim-Nasir",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rabih-Halwani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Raed-A-Al-Qawasmeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Raed-Abu-Odeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ramakrishnan-Ramanathan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ramesh-Bansal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rami-Al-Ruzouq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rasha-M-Hattab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rifat-Hamoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ryan-Sean-Bakker",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saad-Harous",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saif-al-Dein-Taha-al-Fuqara",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salah-Altoubat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salahedeen-Abusnana",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saleh-Abu-Dabous",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salim-A-Messaoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salman-Yousuf-Guraya",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sam-Sulaiman-Dalla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samar-Mouakket",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sameh-Al-Shihabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samer-A-Barakat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samih-Mahmoud-AL-Karasneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sane-M-Yagi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shehdeh-Fareh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Soliman-Mahmoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Suhail-Hani-Al-Amad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syed-Amir-Gilani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syed-Awais-Ahmad-Tipu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tahar-Laoui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tamer-Rabie",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tevhide-Serra-Gorpe",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Usha-Ramanathan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wegdan-Bani-issa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/William-C-Frick-Ph-D",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yasser-Khalil-Bustanji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yuhanis-Binti-Ab-Aziz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zaher-Al-Aghbari",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdel-Rahman-Ahmed-Abdel-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdelaziz-EL-Gamouz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdelnaser-Alsayid-Mohamed-Aljahani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abrar-Inayat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abu-Elias-Sarker",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adel-Elmoselhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahcene-Bounceur",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Al-Azayzih",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Qasim-Mohammad-AlHamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Rasdan-Bin-Ismail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Galal-Abokhalil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Hachicha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Hossain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-M-Almehdi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Musa-Hayajneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Yusuf-Alhusban",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Altaii",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Makki",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alham-Jehad-Ali-Al-Sharman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Ahmed-Adam-Ismail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Ibrahim-Saeed-Ahmed-Shorbagi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Mohammed-Hassan-Radwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anu-Vinod-Ranade",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Arif-Mohammed-Aljanahi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aseel-Ali-Hussien",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ashokan-Arumugam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Atif-Awoad-Abdallah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ayat-Jebril-Jaber-Nashwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Azaddin-S-Khalifa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Badeeah-Khaleel-ALHashemi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Balsam-Qubais-Saeed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bashair-Mohammed-Mussa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bassam-Abdullah-Khuwaileh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Betul-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dalila-Berraf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Daniel-Moraetis",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Di-Zhang",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dilber-Uzun-Ozsahin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Emad-Abdel-Hafiz-Ali-Alzyadat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Engy-Khalil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eqab-Almajali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Esam-Saeed-Abed-AlObaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Zohra-Aouati",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatma-Refaat-Abd-El-Fattah-Ahmed-Abd-El-Baky",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fawzia-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Firdos-Ahmad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Gerald-Naughton",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghalia-Abdul-Khader-Khoder",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hadia-Radwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamad-Rashid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamoud-TANNAR",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamzah-Hussein-Abdallah-Elrehail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hasan-Yaser-Alniss",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hassen-Hadj-Kacem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hatem-Mostafa-El-Damanhoury",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hayssam-Dahrouj",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Heba-Abdel-Hamid-Mohammad-Khalil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hussain-Alawadhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Eltayeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ijaz-Ur-Rehman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Abdel-Shahid-Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Akef-Abdelhalim-Khowailed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Akour",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/In-ju-Kim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jacqueline-Dias",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jalal-Taneera",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jamal-barafi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jay-Hetrick",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Joshua-Watts",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kabiru-Goje",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kalyana-Chakravarthy-Bairapareddy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kamis-Younis-Gaballah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kamrul-Hasan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kazifahmida-Farzana",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khaled-Abass",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khaled-Besbes",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khaled-Zamoum",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khalid-Bajou",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khawla-Alnajjar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khayrat-Ayyad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kheireddine-Yousef-Chatra",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Leila-Cheikh-Ismail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/M-Talha-Junaid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maamar-Bentria",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Madou-Gaye-Sylla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahdi-Kais-Abdualkarim-Al-Janabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Ahmed-Albreem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Rabie-Abdel-Hafez",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Ramadan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahreen-Arooj",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Malek-Masmoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Malek-Mohammad-Jamaliah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mamun-Rashid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manar-Wasif-Abu-Talib",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Marwan-Al-Momani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mashhad-Al-Allaf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Massimo-Ragnedda",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Md-Mahfuzur-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Md-Moniruzzama",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mehdi-Jemmali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mehmet-Sukru-Bellibas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mia-Swart",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mian-Muhammad-Ajmal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamad-Ahmad-Shara",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamad-Hamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Abdalla-Nour",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Abdallah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Abuzaid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Dawood-Shamout",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-El-Naggar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Elhoseny",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-G-Arab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Haider",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-I-Abdel-Fattah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Talal-Bonny",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Adel-Moufti",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Ahmad-Qasim-Al-Shabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Alakhrass",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-G-Mohammad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Miftaur-Rahman-Khan-Khadem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Saleh-Alrashdan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Saleh-Bataineh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Amjed-Alsaegh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Hersi-Warsame",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Kamil-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohannad-Nassar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mokhtar-Elareshi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Mahmoud-Idelbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Monther-Jamhawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Moudaffar-Al-Rawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhamed-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Arsyad-Subu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Mustafa-Raziq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Saboor",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Tawalbeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Turki-Alshurideh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Usman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Zubair",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Hassan-Hammash",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadia-Rashed-Ali-Al-Mazrouei",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Najib-Ismail-Jarad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Narjes-Haj-Salem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nassir-Bou-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Naveed-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nazzal-Kisswani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nidal-Mohammed-Alzboun",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nora-Guenifa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Norhayati-Zakaria",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noura-Metawa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omaima-Mohamed-Elsayed-Abouelkheir",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Osman-Abul",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Panagiotis-Zervopoulos",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rania-Harati",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Raouf-Fareh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reyhan-Sabri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rizwan-Qaisar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saadat-M-Alhashmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saaid-Ayesh-Al-Shehadat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saber-Elsayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saeed-Abdallah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saleh-Muhammad-Zeki-Mahmood-Al-Lehaibi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salah-Haridy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salem-B-Abdalla-Salem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salman-Yousaf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samar-Abdullah-M-Damiati",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sameera-Setoutah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sameh-Soliman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samir-Mohammed-Dirar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sawsan-Hammad-Salem-Abuhammad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Semiyu-Aderibigbe",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Serene-Adnan-Moh-d-Badran",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shadi-Adnan-Alshdaifat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sharif-Alghazo",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shek-Atiqure-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shereen-Mohammad-Suleiman-Aleidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shishir-Ram-Shetty",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shouib-Nouh-Ali-Ma-bdeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Simon-Badran",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sofiane-Khadraoui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sohaib-Majzoub",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sohail-Abbas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Suha-Al-Naimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sujan-Piya",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syed-Abidur-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syed-Aziz-ur-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tamer-Mohamed-Shousha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tarek-Merabtene",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tareq-Salameh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Venkateshbabu-Nagendrababu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Vidya-Seshan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Vittorino-Belpoliti",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Walaa-ElKelish",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Waleed-Zeiada",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Walid-Kamal-Mohamed-Abdelbasset",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wiam-ELShami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yassir-Ahmed-Abdu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yazid-Abubakar-Abdullahi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yousif-Abdelbagi-Abdalla-Omer",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zafar-Said",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zahid-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zahid-Raza",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zaid-A-Al-Sadoon",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zaid-Ali-Zaid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zaid-Ghanem-Hamdoon",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aaesha-Saeed-Majid-Saeed-Almesafri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Samad-al-Khalidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdullah-Al-Mutery",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdulrahman-Abdullatif-Mohammad-Abdullatif",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adewale-Olalekan-Giwa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abubakr-H-Mossa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Abuhelwa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Mazhar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Al-Makky",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-M-Aziz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Abdullah-Al-Mutawa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ala-Altaweel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alfan-Saghayir-Mubarak-Markhan-Alketbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Younis-Omar-Al-Muhammad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alounoud-Mohamed-Hassan-Salman-Al-Marzouqi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alya-Alnuaimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Hussein",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Ibrahim-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ameena-Abdullah-Ali-Al-Katri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amjad-Alhalaweh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ammad-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Khalid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anirudh-B-Acharya",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Antonios-Manousakis",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aouatef-Zerara",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aref-Maksoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asima-Karim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Almansoori",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Hamdan-Alsaadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asmaa-Mohamed-Ahmed-Nusairi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Atia-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Awni-Mohd-Saleh-Kasawneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ayad-Turky",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aziz-Farhan-M-Alenezi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aziz-Takhirov",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Badreyya-Ali-Alshehhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Badria-Mohammed-Salem-Al-Hoolah-Al-Shamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bassel-Al-Homssi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bilal-Ahmed-Arain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bouziane-Brik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Chiraz-Anane",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Concetta-SEMERARO",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Danilo-Dessi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dr-Nada-Jamal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/ElSayed-Emad-Ahmed-Nosair",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eman-Ahmed-Abdul-Rahman-Abdullah-Al-Abdouli",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Erhan-Bogan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eslam-Nofal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatemeh-Saheb-Sharif-Askari",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Feras-Jassim-Jirjees",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fikry-El-Naggar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghada-Siddig-Abdin-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hafsa-Khurshid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hala-Georges",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Halima-Khalid-Almidfa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamdan-Hamdan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Haydar-A-Hasan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hisham-Yehia-El-Batawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hussien-Ali-Hussien",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Abaker-Hashem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Yaseen-Hachim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ihsanullah-Obaidullah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Imad-Afyouni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Iman-Khamis-Alyahyaee",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Irene-Pasina",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Isam-Al-Jawarneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ismail-Ben-Douissa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jenna-Saud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kaltum-Mohamed-Hared",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kais-Belwafi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Karam-Mohamed-Sallam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Karima-Al-Shomely",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khadija-Mohamed-Ali-Malik-Aldhuhoori",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khalid-Ali-Altirkawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khalid-Awad-Al-Kubaissi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khalid-Javeed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Leila-Labidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Lina-Abu-Nada",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Batal-Mohamed-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mais-Medhat-Sadek",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Majd-Musa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manal-Mirza-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Marwan-Mansoor-Ali-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mazin-Husain-Hariri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Meeyoung-Kim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mejd-Almheiri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mian-Ahmad-Jan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Ahmed-Eladl",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Al-naqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Hassan-Taha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Ibrahim-Madkour-Zaher",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Shaban-Nafie-El-Sayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Yahya-El-Kishawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Yousif-Al-Hammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Owais-Farooqui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Yousef-Al-Haik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Lataifeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Montaha-Anjass-Almasri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Moohammed-Wasim-Yahia",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Moyassar-Zuhair-Al-Taie",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muamer-Abuzwidah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Al-Mahameed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Mubashhir-Shaikh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammed-Ayas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Alwasmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Munira-Mohammed-Abdullah-Salem-Rahmah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nada-Abdallah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadia-Alkalbani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Najeh-Rajeh-Ibrahim-Alsalhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Narjes-Saheb-Sharif-Askari",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nashwa-Ahmed-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nawal-Askar-Al-Naqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nihar-Dash",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noha-Ahmed-Mousa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nouf-Aljasmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noura-Nasir-Salim-Musfar-Alkarbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omaima-Alqassimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omar-Daoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ousama-Lazkani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Pieter-Willem-Grobbelaar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rabah-ALmahmoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ram-Mantha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rania-Al-Sabbagh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ray-Saadaoui-Mallek",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reda-Ibrahim-Abdelgalil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Abdulla-Alhajji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Richard-Mottershead",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ridvan-Aydin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Roula-Ahmad-Maya",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ruqayya-Mohammed-Ahmed-Bin-Saleem-Alteneiji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saddaf-Rubab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Safeya-Obaid-Mohamed-Almazrouei",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salama-Mohamed-Al-Rahoomi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salima-Hamouche",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samer-Jarbou",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sami-Luigi-De-Giosa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sami-Sulieman-Al-Qatawneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sanaa-Benmessaoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sangeetha-Narasimhan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaojin-Chai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shareefa-Al-Marzooqi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sharifa-Mohamed-Saif-Abdelrahman-Alsuwaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sheer-Abbas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Smriti-Aryal-AC",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Snigdha-Pattanaik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sourjya-Shorjo-Bhattacharjee",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sunaina-Shetty-Yadadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Syed-Ali-Hussain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Taewan-KIM",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tahir-Abbas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tanveer-ul-Haq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Thanh-Mai-VU",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Thomas-Dale-Stollar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Venkata-Veera-Muralee-Gopi-Chandu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Waad-M-Kheder",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wafa-Alnakhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/WAFA-BEN-AMOR-BARHOUMI",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Waqar-Ahmed-Khan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yahya-Haseeb-Yahya-Dallal-Bashi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zainab-Mohamed-AL-Shareef",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zehra-Canan-Araci",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zelal-Jaber-Kharaba",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zia-ul-Haq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Khaliq-Humam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdul-Raheem-Khudada-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abier-Abdul-Sattar-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adnan-Al-Bustanji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Manar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Manar-Laham",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Sukkar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alexandria-Stoya-Zlatar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-AbdulQader-Al-Qabbani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Elbakri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Hasan-Al-Alaimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Tahmaz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alina-Erimia",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Ahbouch",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anas-Abbas-Eidan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Andrea-Lonhardt",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asil-Adil-Al-Baghdadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asil-AlBaghdadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Al-Naqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ayesha-Muhammad-Talha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Azeera-Abdul-Rahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ban-Al-Joubori",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Basheer-M-J-Salman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bassam-Rashed-Khader",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dalal-Abdulmohsen-ALRossais",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dame-Toure",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dana-Faraj-Salahat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dana-Khalil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Deepika-Kamath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Elaf-Akram-Al-Zubaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eman-Al-Hashimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Emenyeonu-Ogadimma",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Faiza-Shadoud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fathima-Afra-Mohiddin-Shaikh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-A-D-Algharbawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Zaid-Mohamed-Saqar-Aldhuhoori",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Gayathri-Arumughan-Kanu",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Georgina-Abood",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghaida-Kaziha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hadil-El-Ankouni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamadeh-Tarazi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hamda-Ebrahim-Mohammed-Saeed-Alawadhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hiba-Jawdat-Barqawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hilda-Allam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Abdel-Karim-Abdel-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Mahmood-Aziz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Inaam-Hamadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Islam-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khadija-Ahmed-Al-Balushi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kholoud-Mustafa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Leena-R-David",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Madiha-Jamil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maged-Elsheshtawy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmod-Abu-Shammeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manal-Mohammed-Shihata-Al-Sha-rawy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Marta-Bialko",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maysara-Adnan-Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mini-Sara-Abraham",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mirhan-Mostafa-Gamal-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Moath-Mheidat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Al-Hemairy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Hussein-Elsayed-Abdelghany-Khater",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Hamza-Mansour",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Khasawneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Yousaf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Hashim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Kanj-El-Harakeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Reza-Salmanpour",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mouza-Jamal-Lootah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Asad-Iqbal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Shahid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Altamimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Eisa-Mohamed-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Nasser-Khadhim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mustafa-Muhamad-Habeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mutwakil-Ibrahim-Ismail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nabiha-Belkacem-Remmani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nada-Mohamed-Al-Hammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadia-Al-Badri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Najla-Abdelwahid-Mohamed-Alzarooni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nasr-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nawal-Nayfeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nida-Siddiqui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nizam-Abdullah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nizar-Mannai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noha-Mousaad-Taha-Elemam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nouf-Alteneiji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omar-Adwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omar-Mahmood-Chebbo",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omer-Jawad-Alali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omer-Shalal-Habeeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Racha-Al-Khoury",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rana-Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saad-Wahby-Al-Bayatti",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sachin-Chaudhary",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saja-Ibrahim-Abdulhadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sama-Abdul-Haq-Suliman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sara-Meer",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sara-Radi-Jamil-Jaser",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sarah-Darwish",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sarra-Ibrahim-Saeed-Shorbagi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saud-Abdalla-Mubarak-Mohamed-Alyafei",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shadi-Alkhatib",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaima-Ahmed-Abdalla-Abdelrahman-Dheyab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sheela-B-Abraham",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shijna-Kappally",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sivapriya-Ramakrishnan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sondos-Abd-Erraheem-Harfil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Suhair-Abu-Hantash",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sumana-Hossain",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Suni-Ebby",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tahani-Alsarayreh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tahira-Amir",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tasneem-Obaid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tor-Seidel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Usha-Rani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Uzma-Inayat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zeina-Hussain-Salih-El-Doory",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aaesha-Ahmed-Al-Mehrezi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abd-Elmahmoud-Elgaili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdel-Rahman-Amarneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdulhadi-Taysir-Al-Salti",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdullah-Abdulkarim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdullah-Mustafa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abdulrahman-Zainal-Alhammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Abigail-Ronald-Louw",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Adeeb-Babeker-Abdalrahim-Babeker",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Afnan-Abdalla-Bin-Yarouf-Alnaqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahlam-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahlam-Al-Khayyal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Ismaiel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmad-Qasem-Ababneh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ahmed-Osama",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aicha-Echtibi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Abdalla-Al-Marzooqi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Al-Hammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Mohammed-Al-Awadhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Rashid-AlShamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Aisha-Rashid-Saud-Ali-Alalawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ala-Abdalla-Alkhaaldi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Abdulhadi-Salmeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Fouad-Alhamarna",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Hassan-Bihi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alaa-Mourhaf-Izzaldin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Al-Samarrai",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ali-Banan-Jamil-Zibdeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alya-Ahmad-Yasser-Radhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alya-Yousif-Sulaiman-Abdlla-Alhammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Alyazia-Obaid-Hassan-Almarashda",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Abdulla-Al-Hadrami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Eltayeb-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Hamza",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amal-Mohammad-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amani-Al-Bawab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amel-G-Hamzeh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amel-Mohamed-Al-Amiri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amel-Yousif-Al-Raeesi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amenah-Hisham-Youssef-Abdelazim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amer-Yousif",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amin-Hasan-Botmah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amina-Al-Boloshi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amina-Jasim-Hassan-AlAli",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amira-Ali-Mohamed-Alkaram",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amira-El-Hamdani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amjad-MHD-Atef-Alhenawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Al-Suwaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Mahmoud-Mohamed-Almulla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Mohamed-Ali-Mohamed-Alsalami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Mohammed-Al-Mousa-Al-Hammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Othman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Salim-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Amna-Shemal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Anas-Cherkawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Arwa-Waleed-Fikri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Arwa-Yousif-Alnaqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Abdalla-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Ahmed-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Aljnebi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Hassan-Alhammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Mohamed-Abdi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Mohammed-Al-Falasi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Mohammed-Qasem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Asma-Nasir-Almakhzoumi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Assiya-Laib",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Badiaa-Alayoubi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Badria-Ahmed-Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bahiya-Ismail-Khalaf",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bashar-Ayman-Amin-Jarah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Basheer-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Batoul-Majed-Karazoun",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Belal-Al-Mashni",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Bento-Joseph",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Buthaina-Humaid-Qatami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Chefaa-Bahaeddin-Alhourani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dalal-Hassan-Mustafa-Al-zubi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dalia-Ibrahim-Hemdan-Ibrahim-Hussein",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dalia-Koraiem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dawood-Salman-Dawood-Alshetiwi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dhabiah-Saif-Saleh-Almutawa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Dina-Hejji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eiman-Khalfan-Ali-Alsaadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eman-Al-Shaibani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Eman-Mohamed-Alketbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Essam-M-Salam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Faheema-Abdalla-ALAli",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Farah-Nihad-Al-Daghistany",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Farman-Khan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Faten-Raafat-Abdelgawad-Elgharib",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Al-Kutbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Ali-Hattawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Bin-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Mohamed-Abdalla-Yousif-Abdall",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fatima-Mohamed-Alyassi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fayiz-Kandakkeel",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Fazlur-Rahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Forat-Mustafa-Almaz-oum",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/G-Abdelhady",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Gene-Soriano",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ghadir-AlNajar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Haidar-Al-Samarraie",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hajar-Ismaeil-Al-Housani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hajir-Ibrahim-Abdalla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hala-Hussein",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hala-Mohammed-Mushtaha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hala-Yahia-Maher",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hana-Elkheir-Hamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hanaa-AbdElhamed-Anwar-Elshahawy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hanadi-Nasr-Almheiri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hanalory-Nofal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hanan-Yousef",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hanin-Kassem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hashim-AL-Hashmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hassan-Vakani-Tariq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Haya-Mohamed-Khamis-Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hayath-Ahmad-Mansour",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Haydy-Nassar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hayfa-Khalil-Bassol",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hessa-Almudharreb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hiba-Al-Sadek",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hiba-Saad-Al-Dagistani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hilda-Rego",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hind-Abdalla-Alrousi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hind-Ahmed-Alzaabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Hissa-Alshamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huda-Ahmed-Al-Ameri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huda-Alhajri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huda-Hussien",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huda-Najeeb",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Huda-Qayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ibrahim-Mohammad-Mushtaq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Imtinan-Attili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jalal-Ahmad-Mostafa-Alzgoul",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jamila-A-Al-Hosany",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Janisha-Kavumpurath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Joey-Echegorin-Abiertas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jubran-Sawaleh-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Jumana-Mohamed-Fouad-Al-Salloum",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kali-Bahadur-Budhathoki",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kawther-Meslem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khadija-Abdolatif-Shamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khadija-Al-Housani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khaled-Bin-Sayeed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khawla-Al-Kaabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Khawla-Al-Naqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Kifah-Al-Taqaz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Lama-Abdulrahim-Abdulmoti",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Lamya-Husain-Taha-Alaydaroos",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Latifa-Mohamed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Lojinah-Abed-Alhameed-Alnobani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Loubna-Chaabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maathir-Salah-Elshafie-Abdelrahman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mada-Talal-Daghistani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Madhu-Keezhilath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maen-Omar-Asaad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maha-Ahmad-Bader",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahiba-Alhammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maha-Alaa-Eddin",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahmoud-Samy-Madi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mahra-Ali-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mais-Juan-Abdalla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maitha-Abdalla-Al-Hosani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maitha-Abdulla-Almazrouei",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maitha-Al-Tamimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maitha-Ali-Hareb-Shagher-Alshuweihi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Majd-Rashid-Al-Mualla",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maktoom-Alahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Manal-Abbas-Mohamed-Ahmed-Abdelsalam",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maria-Theresa-Abellana-Aqui",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-AlQassimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Obaid-Khalifa-AlKaabi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Rashed-Al-Naqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Saho",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Salim-Alqaydi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mariam-Yousif-Alhammadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Marwa-Al-Hashmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maryam-Butti-Saeed-Alshamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mashael-Mohammed-AlNaqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mawadah-Mubarak",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/May-Tamim-Mohammed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Maytha-Abdullah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mazen-Al-Samman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Meera-Hejji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Meera-Obaid-Al-Salami",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mena-Moyassar-Al-Mallah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamad-Abdallah-Mohamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Abdelmohsen-Mohamed-Sayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohamed-Ammar-Abo-Jouma",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Farooq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammad-Saad-Suleiman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Al-Farouq",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Mohammed-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Qaisieh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mohammed-Zia-ur-Rahman-Khan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Ibrahim-Musa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-M-Al-Dajani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Mohamed-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mona-Naseer-Mahfood",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Monia-Hassan-El-Hajj",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Morad-Mohammad-Al-Adaelah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mouza-Saif-Alyileili",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muatasim-Alkbaisy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Amjad-Maqsood",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Mohsan-Parvaiz",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muhammad-Umar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Abdul-Rahim-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Muna-Hassan-Albelooshi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Munira-Ali-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mussab-Osama-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Mustafa-Snoubra",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nabeil-Salah-Fathy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nabila-Hawawini",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nabila-Hussein",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nada-Othman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nadya-Al-Qayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Najeeba-Ahmad-Karrani",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Najwa-Al-Qaseer",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nasima-Mohamed-Al-Yassi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nasser-Zahra",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Navas-Valiya-Malayil",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nawal-Al-Khzaimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nawal-Al-Saadi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nemat-Dek-Al-Bab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noor-Ul-Misbah-Khanum",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noora-Al-Bloushi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noora-Majed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Nour-Ali-Kousa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noura-Ali-Omran",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noura-Khaiwa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noura-Omran",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Noushad-Pandikadavath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Obaida-Ali-Abu-Bader",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omar-Hassan-Omar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Omar-Mohamed-Kamal-Ahmed-Gouda",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Oroob-Husam-Rashid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Osama-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Osama-Taqatqa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Polite-Mangoro",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Priya-Kaimal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rabab-Tagelsir-Osman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Radhiya-R-Al-Rajaby",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Raheesa-P-Kader",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rana-Shaker-Khalid",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rania-Abdullah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rania-Salaheldein-Osman-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ranya-Salem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rasha-Adnan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rasha-Basil-Saffarini",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rawda-Al-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Razan-Bassim-AL-Humaidi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reed-AlMail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Al-Jaber",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Al-Mashat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Ali-Alkhatri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Mohamed-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Rashid-Alteneiji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zoya-Zahur-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zeinab-Ibrahim-Ali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zeinab-Abdallah-Ibrahim-Hassan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zainab-Rashid-Al-Buraimi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zainab-Barem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yousef-Shaker",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Yusra-Ahmed-Mohammed-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Zainab-Ahmed-Suliman",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Taghreed-Abduallh",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Tarek-Alhomsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Thaeir-Helal",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Thuraya-Al-Ban",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Umair-Ilyas-Malik",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Unaiza-Ismail-Riyad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Wijdan-Atitalla-Jubara-Hamad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Vida-Abdolhamid-Salmanpour-Al-Awadhi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shomous-Abd-Elwahab-Nugud",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sidhik-Mohammad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sol-Andrew-Domingo",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sondus-Al-Qudah",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Suad-Hassan-Elmi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sumayah-Al-Jaede",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sumayia-Anwar-Alhmoudi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sura-Majid-Mohd",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Mubarak-Khamis",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaimaa-El-Shamy",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaista-Manzoor",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shajahan-Andathode",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shamma-Al-Khatri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shamma-Salim-Khamis-Khalfan-Alketbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sharifa-Nabeel-Al-Balooshi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shatha-Khalifa-Saif-Alyammahi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shafeeque-Puthiya",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Abdalla-Alnaqbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Abdullah-Al-Qaydi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Ahmed-Al-Mansoori",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Al-Shamsi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Juma-Obaid-Khamis-Albedwawi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Shaikha-Mohamed-Ali-ALKetbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samar-Elsayed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samar-Naser-L-Alhattab",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samiha-Baniyas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Samira-Saba",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sanoor-Mannath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sara-Ali-Dalli",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sara-Atef",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sarah-Abdulla-Eissa",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Safa-Abdulkader",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Sahar-Saleem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Said-Shahwan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salamah-Al-Kitbi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salem-Khaled-Salem",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salma-Abu-Qiyas",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/SALMA-MOHAMED-IBRAHIM",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Salwa-Mohamad-El-Nassar",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Renji-Mathews",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rim-Helali",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Roba-Mohamed-Maher-Nassif",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rola-Jamal-Abu-Farha",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rouf-Thekkepurath",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Rozan-Awad",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Ruwaya-Khalfan-Saif-Almesafri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Saad-Zafarul-Hasan",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reed-AlMail",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Al-Jaber",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Al-Mashat",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Ali-Alkhatri",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Mohamed-Ahmed",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Rashid-Alteneiji",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Reem-Saeed-Alqaydi",
            "https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff/Renira-Nisha-Lobo"
            ]
            
# for x in prof_urls: 
#     data = extract_prof_info(x)
#     print(data)

Prof. Abbes Amira(Dean). Academic Rank:Professor. Specialization:Ph.D. in Computer Science (Artificial Intelligence and IoT Applications), Queen’s University Belfast, UK. Faculty details: mailto:aamira@sharjah.ac.ae,. Faculty details: tel:+(971) 6 5050525,. Faculty details: https://www.researchgate.net/profile/Abbes-Amira-2,. Faculty details: https://scholar.google.co.uk/citations?user=yp_wQZMAAAAJ&hl=en,. Faculty details: https://www.scopus.com/authid/detail.uri?authorId=23003399100,. Faculty details: https://www.sharjah.ac.ae/-/media/project/uos/sites/uos/academics/faculty-and-staff/cv/abbes-amira.pdf,. 
Dr. Hussein M. Elmehdi(Dean of Academic Support Services). Academic Rank:Associate Professor. Faculty details: mailto:hmelmehdi@sharjah.ac.ae,. Faculty details: tel:97165050359,. Faculty details: https://www.researchgate.net/profile/Hussein_Elmehdi,. Faculty details: https://scholar.google.com/,. Faculty details: https://www.scopus.com/authid/detail.uri?authorId=6508237128,. Faculty 